# Phase 5 — Causal cross-ablation, both organisms (generation only)

See `docs/research_proposal.md` §4.6. Geometry alone is not sufficient evidence -- this is the test that matters. Generates completions under 4 ablation conditions per organism (120 completions/organism, 240 total), resumable the same way Phase 2 was. Grading and score-delta computation now happen locally (`grade_ablation_local.ipynb`), not here -- same reasoning as Phase 3's move off Colab: no GPU needed for grading, keep GPU time for GPU-only work.

In [ ]:
%pip install -q unsloth peft transformers trl datasets huggingface_hub accelerate bitsandbytes pillow pyyaml anthropic
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_DIR = '/content/drive/MyDrive/emergent-misalignment-project'  # upload src/ here
sys.path.append(PROJECT_DIR)
import os
os.chdir(PROJECT_DIR)  # src/ modules use paths relative to the project root (e.g. data/eval/scenarios.json)
ARTIFACTS = f'{PROJECT_DIR}/artifacts'


## Load the curated multimodal eval set (same fixed set as notebook 03)

In [ ]:
from src.generate import load_multimodal_eval_set

mm_set = load_multimodal_eval_set()
mm_images = [ex['image'] for ex in mm_set]


## Run the four ablation conditions per organism

Resumable per-condition (`gen_{condition}_{organism}.jsonl`, tqdm progress bar) via the same `generate_completions` used in Phase 2. Frees GPU memory between organisms since two checkpoints get loaded sequentially -- same pattern as notebook 02.

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import torch

from src.ablation import run_cross_ablation
from src.train import load_base_model

LAYER = 20

for organism in ['A', 'B']:
    completions = json.loads((Path(ARTIFACTS) / f'phase2_completions_{organism}.json').read_text())
    direction_text = np.load(f'{ARTIFACTS}/direction_text_{organism}.npy')
    direction_mm = np.load(f'{ARTIFACTS}/direction_mm_{organism}.npy')

    checkpoint = Path(f'{ARTIFACTS}/checkpoint_path_{organism}.txt').read_text().strip()
    ft_model, ft_tokenizer = load_base_model(checkpoint)

    ablated = run_cross_ablation(
        ft_model, ft_tokenizer, direction_text, direction_mm, LAYER,
        text_prompts=completions['text_prompts'], mm_prompts=completions['mm_prompts'], mm_images=mm_images,
        organism=organism, out_dir=ARTIFACTS,
    )

    aggregate = {
        name: {
            'prompts': completions['mm_prompts'] if 'mm' in name else completions['text_prompts'],
            'base_completions': completions['mm_base_completions'] if 'mm' in name else completions['text_base_completions'],
            'ablated_completions': ablated_completions,
        }
        for name, ablated_completions in ablated.items()
    }
    Path(f'{ARTIFACTS}/phase5_generations_{organism}.json').write_text(json.dumps(aggregate))
    print(f'organism {organism}: done,', {name: len(v) for name, v in ablated.items()})

    del ft_model
    gc.collect()
    torch.cuda.empty_cache()


## Fallback if cross-ablation is weak/null -- orthogonal-projection variant (SARSteer)

Only run this (per organism, as needed) if `cross_mm`/`cross_text` deltas (computed locally, see `grade_ablation_local.ipynb`) are near zero relative to their `within_*` baseline, before concluding H0.

In [ ]:
from src.ablation import orthogonal_projection_ablation
from src.directions import get_activations

# distributional_gap = mean(mm activations) - mean(text activations) at LAYER, from Phase 4 activations
# with orthogonal_projection_ablation(ft_model, direction_text, LAYER, distributional_gap):
#     retry_completions = generate_completions(ft_model, ft_tokenizer, completions['mm_prompts'], images=mm_images)


## Next step

Download `phase5_generations_A.json` and `phase5_generations_B.json` from Drive's `artifacts/` into local `artifacts/`, then run `grade_ablation_local.ipynb` (grades all 4 conditions x 2 organisms and computes score deltas vs. the Phase 3 baseline, entirely locally).